In [14]:
import polars as pl

### Load the datasets:
- ADA - Area of Activity (professional occupation)
- Attivita - Skills related to Area of Activity (many to one)
- CP2021 - Classification of Professions (more granular than Areas of Activity)

In [15]:
ada = pl.read_csv("../data/rtc/ada_cp2021.csv", separator=";")
attivita = pl.read_csv("../data/rtc/attivita_ada.csv", separator=";")
cp2021 = pl.read_csv("../data/rtc/five_digit-cp2021.csv", separator=";").select(pl.col(["codice_cp2021", "cp2021_name", "cp2021_desc"]))

In [16]:
ada = ada.join(cp2021, on='codice_cp2021', how='left')
del cp2021

### Load the OJPs datasets

In [4]:
ojp = pl.read_parquet("../data/lightcast/ITA_2025_postings.parquet")
#ojp_pd = pd.read_csv("../data/lightcast/ITA_2025_postings.csv")

# Only keep postings in Calabria (ITF6)
ojp_calabria = ojp.filter(pl.col("LAA_ADMIN_AREA_1") == "ITF6")
del ojp

In [5]:
# Get unique IDs from ojp_calabria
calabria_ids = ojp_calabria.select("ID").unique()

# Filter skills_all lazily
skills_calabria = skills_all.lazy().join(
    calabria_ids.lazy(),
    on="ID",
    how="inner"
).collect()

del skills_all
del calabria_ids

In [6]:
ojp_calabria_join = skills_calabria.join(ojp_calabria, on='ID', how='left')